# Evaluation Metrics Pipeline

This notebook runs an end-to-end evaluation workflow for two RAG strategies:

- **Strategy A**: Hybrid Retrieval (BM25 + Vector)
- **Strategy B**: Hybrid Retrieval + Rerank

It generates predictions, computes metrics against ground truth, and summarizes results.

## Metrics
- Retrieval: Recall@K, Precision@K, MRR@K, nDCG@K, MAP@K
- Generation: Exact Match, Token F1, ROUGE-L
- Citation (if available): Citation hit rate

In [5]:
from pathlib import Path
import subprocess
import json

try:
    import pandas as pd
except Exception:
    pd = None

def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for cur in [p] + list(p.parents):
        if (cur / "config.py").exists() and (cur / "scripts").exists() and (cur / "data").exists():
            return cur
    raise RuntimeError("Cannot find repository root")

ROOT = find_repo_root(Path.cwd())
print("Repository root:", ROOT)

Repository root: /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax


## 1) Install dependencies

Run once if needed.

In [6]:
import sys

def run(cmd):
    print("\n$", " ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

# always use the same Python as the running notebook kernel
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

# install conflict-prone core deps first
run([
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "openai==1.66.0",
    "google-genai==1.66.0",
    "pydantic>=2.9.0,<3.0.0"
])

# then install the rest from absolute requirements path with eager upgrades
run([
    sys.executable, "-m", "pip", "install",
    "--no-cache-dir",
    "--prefer-binary",
    "--upgrade",
    "--upgrade-strategy", "eager",
    "-r", requirements_file
])


$ /opt/anaconda3/bin/python -m pip install --upgrade pip setuptools wheel

$ /opt/anaconda3/bin/python -m pip install --no-cache-dir openai==1.66.0 google-genai==1.66.0 pydantic>=2.9.0,<3.0.0


NameError: name 'requirements_file' is not defined

## 2) Configure input/output paths

In [7]:
GT = ROOT / "data" / "qa_pairs" / "eval_ground_truth.jsonl"
if not GT.exists():
    print(f"Ground truth file not found, generating: {GT}")
    gen_script = ROOT / "scripts" / "eval" / "generate_eval_ground_truth.py"
    if not gen_script.exists():
        raise FileNotFoundError(f"Ground truth generator script not found: {gen_script}")

    subprocess.run(
        ["python", str(gen_script), "--out", str(GT)],
        cwd=ROOT,
        check=True
    )

    if not GT.exists():
        raise FileNotFoundError(f"Failed to generate ground truth file: {GT}")
    raise FileNotFoundError(f"Ground truth file not found: {GT}")


PRED_HYBRID = ROOT / "outputs" / "preds_hybrid.jsonl"
PRED_HYBRID_RERANK = ROOT / "outputs" / "preds_hybrid_rerank.jsonl"

EVAL_HYBRID = ROOT / "outputs" / "eval_hybrid.json"
EVAL_HYBRID_RERANK = ROOT / "outputs" / "eval_hybrid_rerank.json"

EVAL_GEN_HYBRID = ROOT / "outputs" / "eval_gen_hybrid.json"
EVAL_GEN_HYBRID_RERANK = ROOT / "outputs" / "eval_gen_hybrid_rerank.json"

print("Ground truth:", GT)

Ground truth: /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/data/qa_pairs/eval_ground_truth.jsonl


## 3) Generate predictions for both RAG strategies

In [8]:
# Strategy A: Hybrid Retrieval (BM25 + Vector)
run([
    "python", "scripts/eval/run_eval_benchmark.py",
    "--gt", str(GT),
    "--out", str(PRED_HYBRID),
    "--enable-rerank", "false"
])

# Strategy B: Hybrid Retrieval + Rerank
run([
    "python", "scripts/eval/run_eval_benchmark.py",
    "--gt", str(GT),
    "--out", str(PRED_HYBRID_RERANK),
    "--enable-rerank", "true"
])


$ python scripts/eval/run_eval_benchmark.py --gt /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/data/qa_pairs/eval_ground_truth.jsonl --out /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/outputs/preds_hybrid.jsonl --enable-rerank false
[debug] QAAgent.__init__ signature: (self, retriever, llm_service, validator, config: Dict)
[debug] cfg_dict keys (48): ['ACTS_CHUNKED_DIR', 'ACTS_CSV', 'ACTS_HTML_DIR', 'ACTS_MD_DIR', 'BM25_B', 'BM25_K1', 'CHUNK_OVERLAP', 'CHUNK_SIZE', 'COHERE_API_KEY', 'COHERE_RERANK_MODEL', 'DATA_DIR', 'DEBUG', 'DOCS_DIR', 'EMBEDDING_DIMENSION', 'EMBEDDING_MODEL', 'EMBEDDING_PROVIDER', 'ENABLE_KG', 'ENABLE_RERANK', 'FINANCIAL_TEMPLATES_DIR', 'GEMINI_EMBEDDING_MODEL', 'GEMINI_LLM_MODEL', 'GOOGLE_API_KEY', 'KG_BOOST_WEIGHT', 'KG_MAX_EXPANSION', 'LLM_MAX_TOKENS', 'LLM_MODEL', 'LLM_PROVIDER', 'LLM_TEMPERATURE', 'LOGS_DIR', 

Error in batch embedding: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
Vector search failed, fallback to BM25/keyword: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
Error in batch embedding: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
Vector search failed, fallback to BM25/keyword: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-00

CalledProcessError: Command '['python', 'scripts/eval/run_eval_benchmark.py', '--gt', '/Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/data/qa_pairs/eval_ground_truth.jsonl', '--out', '/Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/outputs/preds_hybrid.jsonl', '--enable-rerank', 'false']' returned non-zero exit status 1.

## 4) Compute evaluation metrics (retrieval + generation)

In [ ]:
run([
    "python", "scripts/eval/evaluate_predictions.py",
    "--gt", str(GT),
    "--pred", str(PRED_HYBRID),
    "--k", "5",
    "--out", str(EVAL_HYBRID)
])

run([
    "python", "scripts/eval/evaluate_predictions.py",
    "--gt", str(GT),
    "--pred", str(PRED_HYBRID_RERANK),
    "--k", "5",
    "--out", str(EVAL_HYBRID_RERANK)
])

## 5) Optional: generation-only metrics

In [ ]:
run([
    "python", "scripts/eval/evaluate_generation.py",
    "--gt", str(GT),
    "--pred", str(PRED_HYBRID),
    "--out", str(EVAL_GEN_HYBRID)
])

run([
    "python", "scripts/eval/evaluate_generation.py",
    "--gt", str(GT),
    "--pred", str(PRED_HYBRID_RERANK),
    "--out", str(EVAL_GEN_HYBRID_RERANK)
])

## 6) Load and compare results

In [ ]:
def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

hybrid = load_json(EVAL_HYBRID)
hybrid_rerank = load_json(EVAL_HYBRID_RERANK)

print("Hybrid:")
print(json.dumps(hybrid, indent=2, ensure_ascii=False))
print("\nHybrid + Rerank:")
print(json.dumps(hybrid_rerank, indent=2, ensure_ascii=False))

In [ ]:
def flatten_eval(label, obj):
    row = {"strategy": label}
    row.update(obj.get("retrieval", {}))
    gen = obj.get("generation", {})
    row.update({f"gen_{k}": v for k, v in gen.items()})
    return row

rows = [
    flatten_eval("Hybrid", hybrid),
    flatten_eval("Hybrid+Rerank", hybrid_rerank),
]

if pd is not None:
    display(pd.DataFrame(rows))
else:
    print(json.dumps(rows, indent=2, ensure_ascii=False))

## 7) Reproducibility artifacts

Keep these files in your report submission:

- `outputs/preds_hybrid.jsonl`
- `outputs/preds_hybrid_rerank.jsonl`
- `outputs/eval_hybrid.json`
- `outputs/eval_hybrid_rerank.json`
- `outputs/run_snapshot_YYYYmmdd_HHMMSS.json` (auto-generated by benchmark script)